# Model Understanding Template
transformer의 구조와 중간 흐름을 이해하기 위한 기본 포맷

#### Goals
- architecture 파악
- attention, query, proposal, feature map 같은 중간 표현 시각화
- 논문 설명과 코드 구조를 같이 읽기

In [2]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

import torch
print("visible:", os.environ["CUDA_VISIBLE_DEVICES"])
print("cuda count:", torch.cuda.device_count())   # 1이어야 정상
print("current:", torch.cuda.current_device())    # 0
print("name:", torch.cuda.get_device_name(0))

visible: 6
cuda count: 1
current: 0
name: NVIDIA B200


In [56]:
import torch
import torch.nn as nn
import math

---

## 0. 구조 개요
- 계열: transformer
- encoder: tokenizer / input embedding / positional enbedding 
- decoder: 

* Transformer: structure / architecture
* Transformere based MODEL
    * Encoder only: BERT, DistilBERT
    * Decoder only: GPT
    * Encoder-Decoder: t5, BART

In [ ]:
transformer = nn.Transformer(
    d_model=768,
    nhead=12,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=3072,
    dropout=0.1,
    activation='relu',
    batch_first=True,
)

# 모델 레이어, 각 레이어 내의 weight
for name, param in transformer.named_parameters():
    print(name, param.shape)

encoder.layers.0.self_attn.in_proj_weight torch.Size([2304, 768])
encoder.layers.0.self_attn.in_proj_bias torch.Size([2304])
encoder.layers.0.self_attn.out_proj.weight torch.Size([768, 768])
encoder.layers.0.self_attn.out_proj.bias torch.Size([768])
encoder.layers.0.linear1.weight torch.Size([3072, 768])
encoder.layers.0.linear1.bias torch.Size([3072])
encoder.layers.0.linear2.weight torch.Size([768, 3072])
encoder.layers.0.linear2.bias torch.Size([768])
encoder.layers.0.norm1.weight torch.Size([768])
encoder.layers.0.norm1.bias torch.Size([768])
encoder.layers.0.norm2.weight torch.Size([768])
encoder.layers.0.norm2.bias torch.Size([768])
encoder.layers.1.self_attn.in_proj_weight torch.Size([2304, 768])
encoder.layers.1.self_attn.in_proj_bias torch.Size([2304])
encoder.layers.1.self_attn.out_proj.weight torch.Size([768, 768])
encoder.layers.1.self_attn.out_proj.bias torch.Size([768])
encoder.layers.1.linear1.weight torch.Size([3072, 768])
encoder.layers.1.linear1.bias torch.Size([3072]

In [42]:
print(transformer)

Transformer(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (linear1): Linear(in_features=768, out_features=3072, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=3072, out_features=768, bias=True)
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicall

---

Pretrained Transformer 계열

## 1. encoder

#### Tokenization / Token Embedding, 토큰화 
* 텍스트를 토큰이라 불리는 작은 단위로 분해하는 과정
* 단어, 서브워드, 문자일 수 있음
* 기계가 텍스트의 개별 요소를 이해하고 처리할 수 있게 함

sentence 1개

In [ ]:
# Sentence 1개 
from transformers import MarianTokenizer, MarianMTModel

tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-de")

text = "I ate tomatoes."

input_ids = tokenizer.encode(text)
tokens = tokenizer.tokenize(tokenizer.decode(input_ids))

print("Tokens:", tokens)
print("Input IDs:", input_ids)
# Marian tokenizer는 고정 단어 사전이 아니라 subword vocabulary를 써서 토큰화
# 덜 자주 나오거나 분해가 유리한 단어는 더 작은 조각으로 나눔

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 19260.82it/s]


Tokens: ['▁I', '▁at', 'e', '▁tomatoes', '.', '</s>']
Input IDs: [38, 67, 18, 36161, 3, 0]


Sentence Pair
* 문장 2개를 하나의 샘플로 - 관계 있는 한 쌍으로 처리
* 문장 관계 추론, entailment, QA, similarity

In [35]:
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

text_1 = "I ate tomatoes"
text_2 = "He ates apples"

input_ids = tokenizer.encode(text_1, text_2)
tokens = tokenizer.tokenize(tokenizer.decode(input_ids))

print("Tokens:", tokens)
print("Input IDs:", input_ids)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3051.40it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens: ['[CLS]', 'i', 'ate', 'tomatoes', '[SEP]', 'he', 'ate', '##s', 'apples', '[SEP]']
Input IDs: [101, 1045, 8823, 12851, 102, 2002, 8823, 2015, 18108, 102]


Sentence Batch
* 문장 여러 개를 독립적인 샘플로 한꺼번에 넣는 것
* 속도 때문에 묶어서 처리

In [36]:
# Scentence 2개 이상
text_a = ["I like tomatoes", "He likes apples"]
text_b = ["Tomatoes tastes good", "Apples are sweet and delicious"]

batch_ids = tokenizer(text_a, text_b, padding=True, truncation=True, return_tensors='pt')
# tokenizer는 같은 인덱스를 pair로 묶어서 batch 만듦 

torch.set_printoptions(threshold=10_000)

for key, value in batch_ids.items():
    print(f"{key:15} | shape: {tuple(value.shape)}")
    print(value)
    print("\n")

# input_ids / token_type_ids / attention_mask 
# token_type_ids 0은 첫 번째 문장, 1은 두 번째 문장

print(f"{'-'*100}")  

for ids in batch_ids['input_ids']:
    tokens = tokenizer.convert_ids_to_tokens(ids)
    print(tokens)

print(batch_ids['input_ids'])
print("Tensor shape:", batch_ids['input_ids'].shape)
# torch.size([2,9]): batch size = 2, seq len = 9

input_ids       | shape: (2, 11)
tensor([[  101,  1045,  2066, 12851,   102, 12851, 16958,  2204,   102,     0,
             0],
        [  101,  2002,  7777, 18108,   102, 18108,  2024,  4086,  1998, 12090,
           102]])


token_type_ids  | shape: (2, 11)
tensor([[0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0],
        [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]])


attention_mask  | shape: (2, 11)
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


----------------------------------------------------------------------------------------------------
['[CLS]', 'i', 'like', 'tomatoes', '[SEP]', 'tomatoes', 'tastes', 'good', '[SEP]', '[PAD]', '[PAD]']
['[CLS]', 'he', 'likes', 'apples', '[SEP]', 'apples', 'are', 'sweet', 'and', 'delicious', '[SEP]']
tensor([[  101,  1045,  2066, 12851,   102, 12851, 16958,  2204,   102,     0,
             0],
        [  101,  2002,  7777, 18108,   102, 18108,  2024,  4086,  1998, 12090,
           102]])
Tensor shape: torch.Size([2, 11])


#### Positional Embedding

* 절대 위치
* sinusodial positional embedding
    * 사인, 코사인 주기 함수로 위치 벡터를 만듦
    * PE(pos, 2i) = sin(pos / 10000^(2i / d_model))
    * PE(pos, 2i+1) = cos(pos / 10000^(2i+1 / d_model))
* final_input = token_embedding + positional_encoding

In [43]:
from transformers import MarianTokenizer, MarianMTModel
tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-de")


text_a = ["I like tomatoes", "He likes apples"]
text_b = ["Tomatoes tastes good", "Apples are sweet and delicious"]

batch_ids = tokenizer(text_a, text_b, padding=True, truncation=True, return_tensors='pt')
print("=== Batch ===")
for key, value in batch_ids.items():
    print(f"{key:15} | shape: {tuple(value.shape)}")
    print(value)
    print()


print("=== Tokens ===")
for ids in batch_ids["input_ids"]:
    tokens = tokenizer.convert_ids_to_tokens(ids)
    print(tokens)

print()
print("input_ids:")
print(batch_ids["input_ids"])
print("Tensor shape:", batch_ids["input_ids"].shape)

print("\n" + "=" * 100)

input_ids = batch_ids["input_ids"]
attention_mask = batch_ids["attention_mask"]

token_emb = model.get_input_embeddings()(input_ids)
pos_emb = model.model.encoder.embed_positions(token_emb.shape[:-1])
combined_emb = token_emb + pos_emb

print("Token embeddings shape:", token_emb.shape)
print("Positional embeddings shape:", pos_emb.shape)
print("Combined embeddings shape:", combined_emb.shape)
print("Positional embeddings:", pos_emb)

/home/jovyan/aicon-gamma-datavol-1/hjgoh/ai-lab/.venv/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 9410.32it/s]


=== Batch ===
input_ids       | shape: (2, 11)
tensor([[   38,   209, 36161,   429, 21191,    97, 32356,   402,     0, 58100,
         58100],
        [  231,   209,     6, 35646,  6262,     6,    48,  7434,     8, 13525,
             0]])

attention_mask  | shape: (2, 11)
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

=== Tokens ===
['▁I', '▁like', '▁tomatoes', '▁To', 'mato', 'es', '▁tastes', '▁good', '</s>', '<pad>', '<pad>']
['▁He', '▁like', 's', '▁apples', '▁Apple', 's', '▁are', '▁sweet', '▁and', '▁delicious', '</s>']

input_ids:
tensor([[   38,   209, 36161,   429, 21191,    97, 32356,   402,     0, 58100,
         58100],
        [  231,   209,     6, 35646,  6262,     6,    48,  7434,     8, 13525,
             0]])
Tensor shape: torch.Size([2, 11])

Token embeddings shape: torch.Size([2, 11, 512])
Positional embeddings shape: torch.Size([11, 512])
Combined embeddings shape: torch.Size([2, 11, 512])
Positional embeddings: tensor([[ 0.0000

In [39]:
with torch.no_grad():
    encoder_outputs = model.get_encoder()(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

print("Encoder last hidden state shape:", encoder_outputs.last_hidden_state.shape)

Encoder last hidden state shape: torch.Size([2, 11, 512])


* 위치 벡터끼리 거리 비교

In [42]:
import torch
import torch.nn.functional as F

# encoder positional embedding table
pos_table = model.model.encoder.embed_positions.weight

# 1번 위치와 2번 위치 비교
v1 = pos_table[1]
v2 = pos_table[2]

l2_dist_12 = torch.dist(v1, v2, p=2)
cos_sim_12 = F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0))

# 1번 위치와 3번 위치 비교
v1 = pos_table[1]
v3 = pos_table[3]

l2_dist_13 = torch.dist(v1, v3, p=2)
cos_sim_13 = F.cosine_similarity(v1.unsqueeze(0), v3.unsqueeze(0))

print("L2 distance:", l2_dist_12.item(), l2_dist_13.item())
print("Cosine similarity:", cos_sim_12.item(), cos_sim_13.item())

L2 distance: 3.7142703533172607 6.966545581817627
Cosine similarity: 0.9730550646781921 0.905209481716156


#### Word Embedding, 단어 임베딩
* 단어를 연속 벡터 공간의 실수로 이루어진 밀도 있는 벡터로 나타내는 것
* 벡터로 매핑, 벡터 간의 거리와 방향이 의미를 가짐
* 단어 간의 의미적 관계를 포착해 기계가 의미와 맥락을 이해할 수 있게 함
* semantic search

Sentence 1개

In [30]:
# load pre-trained tokenizer: DistilBERT tokenizer, model
# 모델 구조 만들고, 학습된 가중치 불러오기 / base model을 메모리에 올림
text = "I ate tomatoes"
encoded_input = tokenizer(text, return_tensors="pt")

input_ids = encoded_input["input_ids"]
attention_mask = encoded_input["attention_mask"]
with torch.no_grad():
    encoder_outputs = model.get_encoder()(input_ids=input_ids, attention_mask=attention_mask)
    outputs = encoder_outputs.last_hidden_state
    # transformer encoder 여러 층 통과 / hidden state 추출 / 문맥 반영 
    # positional encoding 더해진 후 임베딩 
    # 모델 출력 중 첫 번째 값

word_embeddings = outputs
print("World Embeddings:")
print(word_embeddings)
print("Tensor Size:", word_embeddings.size()) 
# torch.size([1, 9, 768]): batch size = 1, seq len = 9, model dimension(d_model) = 768

World Embeddings:
tensor([[[ 1.7574e-01, -9.8591e-02, -2.1946e-02, -6.1331e-02, -6.0537e-02,
           1.7179e-01,  1.7524e-01, -1.6694e-01, -2.8908e-01, -3.9872e-02,
          -2.3384e-02,  2.5485e-01,  3.2568e-01, -5.8895e-02, -4.1906e-02,
          -7.0191e-02, -3.0089e-01,  2.5588e-01, -4.0447e-02, -7.9274e-02,
          -5.2348e-01,  1.3636e-02,  3.3623e-01,  8.4826e-02,  1.5949e-01,
           1.2990e-01,  2.3739e-01,  2.5311e-02,  1.9941e-01, -6.9312e-02,
           1.4300e-01, -1.5552e-01,  9.2513e-02,  1.5886e-01, -3.8854e-02,
           3.0009e-01,  8.7498e-03,  1.9669e-01,  7.1631e-03,  1.1502e-02,
          -8.2838e-02, -6.6549e-02,  1.5992e-01,  6.0511e-03,  4.5231e-01,
          -1.5124e-01, -2.3748e-01, -3.4613e-03, -1.4064e-01, -9.5647e-03,
           5.8282e-02, -3.2016e-02,  1.5573e-01,  1.8274e-01, -9.5261e-03,
           3.9399e-01, -3.2798e-02, -4.3242e-01,  1.5853e-01,  1.7661e-01,
           6.8846e-02,  2.2903e-01,  1.0839e-01,  1.6494e-01,  7.9161e-02,
       

Sentence batch

In [33]:
text_a = ["I like tomatoes", "He likes apples"]
text_b = ["Tomatoes tastes good", "Apples are sweet and delicious"]

batch_input = tokenizer(text_a, text_b, padding=True, truncation=True, return_tensors='pt')
input_ids = batch_input["input_ids"]
attention_mask = batch_input["attention_mask"]

with torch.no_grad():
    encoder_outputs = model.get_encoder()(input_ids=input_ids, attention_mask=attention_mask)

word_embeddings = outputs[0]
print(word_embeddings)
print(word_embeddings.shape)
# torch.size([1, 9, 768]): batch size = 2, seq len = 11, model dimension(d_model) = 768

tensor([[ 1.7574e-01, -9.8591e-02, -2.1946e-02, -6.1331e-02, -6.0537e-02,
          1.7179e-01,  1.7524e-01, -1.6694e-01, -2.8908e-01, -3.9872e-02,
         -2.3384e-02,  2.5485e-01,  3.2568e-01, -5.8895e-02, -4.1906e-02,
         -7.0191e-02, -3.0089e-01,  2.5588e-01, -4.0447e-02, -7.9274e-02,
         -5.2348e-01,  1.3636e-02,  3.3623e-01,  8.4826e-02,  1.5949e-01,
          1.2990e-01,  2.3739e-01,  2.5311e-02,  1.9941e-01, -6.9312e-02,
          1.4300e-01, -1.5552e-01,  9.2513e-02,  1.5886e-01, -3.8854e-02,
          3.0009e-01,  8.7498e-03,  1.9669e-01,  7.1631e-03,  1.1502e-02,
         -8.2838e-02, -6.6549e-02,  1.5992e-01,  6.0511e-03,  4.5231e-01,
         -1.5124e-01, -2.3748e-01, -3.4613e-03, -1.4064e-01, -9.5647e-03,
          5.8282e-02, -3.2016e-02,  1.5573e-01,  1.8274e-01, -9.5261e-03,
          3.9399e-01, -3.2798e-02, -4.3242e-01,  1.5853e-01,  1.7661e-01,
          6.8846e-02,  2.2903e-01,  1.0839e-01,  1.6494e-01,  7.9161e-02,
          1.7642e-01,  6.7107e-02,  4.

#### Text Embedding, 텍스트 임베딩
* 문서 전체를 밀도 있는 벡터로 나타내 텍스트 전반의 의미를 캡처하는 기술
* 문서나 문장을 의미적 공간에서 비교 분석할 수 있게 함
* 텍스트 전체를 하나의 벡터로 표현
* 텍스트 분류, 감성 분석, 검색 엔진 랭킹, 군집 및 유사도 분석, 질의응답 등에 활용

Transformer를 이용한 text embedding
* mean pooling으로 문장 전체를 하나의 벡터로 
* 다른 방법: [CLS] token only, max pooling, attention pooling
* attention mask: 모델이 실제로 봐야 하는 토큰과 무시해야 하는 토큰을 구분하는 마스크 

In [19]:
import torch
import torch.nn.functional as F  

from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

text_a = ["I like tomatoes", "He likes apples"]
text_b = ["Tomatoes tastes good", "Apples are sweet and delicious"]

batch_ids = tokenizer(text_a, text_b, padding=True, truncation=True, return_tensors='pt')

with torch.no_grad():
    outputs= model(**batch_ids)
# torch.size([1, 9, 768]): batch size = 2, seq len = 11, model dimension(d_model) = 768

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] # 첫 번째 출력은 토큰 임베딩 / 토큰별 벡터
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Pooling
sentence_embeddings = mean_pooling(outputs, batch_ids['attention_mask'])

# Normalize embeddings
sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

print(sentence_embeddings)
print("Torch size:", sentence_embeddings.size())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2177.06it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tensor([[ 5.0335e-03,  4.1990e-02, -6.2122e-03,  3.1160e-02,  3.9375e-03,
         -1.1122e-02,  1.8193e-02,  9.6589e-02,  1.1619e-02, -1.7427e-02,
          5.3702e-03, -1.6652e-02,  1.7088e-02,  8.4708e-03, -3.2806e-02,
          1.7244e-02,  7.9869e-03,  1.9216e-02,  8.0551e-03,  5.4825e-02,
          5.7593e-02, -1.5216e-02, -1.5353e-02,  2.1508e-02,  3.7572e-02,
          1.6977e-02,  1.0068e-02,  2.1549e-02, -5.4518e-03, -3.5973e-02,
          1.8568e-02,  1.4838e-02,  7.7869e-03,  4.5616e-03,  2.8350e-02,
         -9.1522e-03, -1.5516e-02, -1.8453e-02, -6.4726e-03,  2.7824e-02,
         -1.9164e-03, -1.3780e-03,  4.7697e-02, -7.0555e-03,  9.1297e-03,
         -5.1256e-02,  1.7619e-02,  4.1116e-02, -6.3143e-03, -2.9965e-02,
          4.0059e-02, -2.2679e-02,  1.0386e-03, -1.6614e-02, -2.1156e-02,
          8.9237e-02, -5.6341e-02, -2.0089e-02, -2.7175e-02,  2.1362e-02,
          5.7799e-04,  3.5522e-03, -1.5354e-02, -2.3096e-02,  1.4286e-02,
          3.6453e-02, -5.0951e-02,  2.

In [21]:
# attention_mask
print("attention_mask:", batch_ids["attention_mask"])

# model_output[0]: 토큰별 벡터
print("model_output[0]:", outputs[0])

attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
model_output[0]: tensor([[[-6.5879e-01,  6.7432e-01, -4.2651e-01,  ..., -8.5615e-01,
           2.7983e-01,  1.3680e-01],
         [-1.2070e-01,  1.8631e-01, -3.0781e-01,  ..., -1.2403e-01,
           5.1486e-01,  2.3350e-01],
         [ 2.2293e-01,  5.3949e-01,  9.8170e-01,  ...,  4.3340e-01,
           7.0316e-03,  1.1633e-04],
         ...,
         [ 8.9028e-01,  6.6665e-02, -4.1428e-01,  ...,  2.4636e-02,
          -5.5317e-01, -4.2066e-01],
         [ 5.5809e-02,  2.5404e-01, -6.6157e-02,  ...,  1.4728e-01,
           2.0025e-01,  3.8939e-02],
         [ 1.4664e-02,  1.5871e-01, -5.2238e-02,  ...,  2.7990e-01,
           2.0061e-01, -6.9511e-02]],

        [[-5.9485e-01,  6.7126e-01, -5.7514e-01,  ..., -4.6078e-01,
           5.5723e-01,  1.9676e-01],
         [ 3.9399e-02, -3.4499e-01, -1.8579e-01,  ..., -2.0947e-01,
           1.1037e+00, -2.8689e-01],
         [ 1.8556e-01,  

sentence-transformers model을 이용한 text embedding

In [37]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

text = "Love is waiting to be loved"
text_embedding = model.encode(text)
print("Text Embedding:", text_embedding)
print("Torch size:", text_embedding.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3703.03it/s]


Text Embedding: [ 9.00400281e-02  2.88154390e-02 -7.06398580e-03 -7.00149918e-03
  1.98615319e-03 -7.74547399e-04 -5.40488996e-02 -1.42236454e-02
 -1.58045720e-02  6.22665463e-03  6.04661042e-03 -7.76759610e-02
  4.34921980e-02 -1.98056325e-02 -4.93592322e-02 -4.42626923e-02
  2.04353535e-04  6.80816472e-02  1.41200311e-02  1.59173587e-03
  4.00266461e-02  4.27113138e-02 -2.19139010e-02 -7.44314073e-03
  1.07810767e-02  1.26945078e-02  2.67028552e-03 -2.36306097e-02
 -3.69781107e-02 -7.77682383e-03 -3.94216143e-02  5.87411830e-03
 -3.97458067e-03 -2.44078748e-02  1.73916271e-06 -6.02714531e-02
  1.10920975e-02 -1.53814033e-02  1.02660013e-02 -1.62619445e-02
 -1.15325293e-02 -5.81199490e-02  9.81204119e-03  2.23624837e-02
 -4.68076728e-02  5.32282516e-02  4.38676141e-02 -3.21675465e-02
 -2.00409871e-02  2.16490552e-02 -1.69524085e-02  1.59915481e-02
 -3.65609974e-02 -4.70665134e-02  6.25194535e-02 -2.42112577e-02
 -2.84350514e-02 -4.47161831e-02  7.44468272e-02  1.23012103e-02
 -2.81453

#### Image Embedding, 이미지 임베딩 

#### Self-attention

* query, key, value
* query: 검색 의도 / key: 태그, 어떤 책을 참고할 지에 대한 기준/ value: 책의 내용 

In [44]:
from transformers import MarianTokenizer, MarianMTModel

# tokenizer, model
tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
model = MarianMTModel.from_pretrained(
    "Helsinki-NLP/opus-mt-en-de", 
    attn_implementation="eager"
)

/home/jovyan/aicon-gamma-datavol-1/hjgoh/ai-lab/.venv/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 24806.99it/s]


In [ ]:
print(model.model.encoder)
print(model.model.decoder)

MarianEncoder(
  (embed_tokens): Embedding(58101, 512, padding_idx=58100)
  (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
  (layers): ModuleList(
    (0-5): 6 x MarianEncoderLayer(
      (self_attn): MarianAttention(
        (k_proj): Linear(in_features=512, out_features=512, bias=True)
        (v_proj): Linear(in_features=512, out_features=512, bias=True)
        (q_proj): Linear(in_features=512, out_features=512, bias=True)
        (out_proj): Linear(in_features=512, out_features=512, bias=True)
      )
      (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
      (activation_fn): SiLU()
      (fc1): Linear(in_features=512, out_features=2048, bias=True)
      (fc2): Linear(in_features=2048, out_features=512, bias=True)
      (final_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
)
MarianDecoder(
  (embed_tokens): Embedding(58101, 512, padding_idx=58100)
  (embed_positions): MarianSinusoi

In [ ]:
encoder_layer0 = model.model.encoder.layers[0] 
# 이미 학습이 끝난 nn.Linear weight을 가지고 있음

print(encoder_layer0)

MarianEncoderLayer(
  (self_attn): MarianAttention(
    (k_proj): Linear(in_features=512, out_features=512, bias=True)
    (v_proj): Linear(in_features=512, out_features=512, bias=True)
    (q_proj): Linear(in_features=512, out_features=512, bias=True)
    (out_proj): Linear(in_features=512, out_features=512, bias=True)
  )
  (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
  (activation_fn): SiLU()
  (fc1): Linear(in_features=512, out_features=2048, bias=True)
  (fc2): Linear(in_features=2048, out_features=512, bias=True)
  (final_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
)


In [ ]:
# text
text_a = ["I like tomatoes", "He likes apples"]
text_b = ["Tomatoes tastes good", "Apples are sweet and delicious"]

# batch tokenization
batch_ids = tokenizer(text_a, text_b, padding=True, truncation=True, return_tensors='pt')
input_ids = batch_ids["input_ids"]      # 정수 인덱스를 vocab embedding table에서 lookup/ 학습된 weight

# scaling: token emd * sqrd(d_madel)
embed_scale = math.sqrt(model.config.d_model) 

# encoder input
with torch.no_grad():
    token_emb = model.get_input_embeddings()(input_ids) * embed_scale
    pos_emb = model.model.encoder.embed_positions(token_emb.shape[:-1])
    hidden_states_0 = token_emb + pos_emb

# self attention
layer_0 = model.model.encoder.layers[0] # 첫 번째 layer 객체 정의 
attn = layer_0.self_attn

# query, key, value
q = attn.q_proj(hidden_states_0)
k = attn.k_proj(hidden_states_0)
v = attn.v_proj(hidden_states_0)

print("Hidden States:", hidden_states_0)
print("Hidden States shape:", hidden_states_0.shape)
print("Q shape", q.shape)   
print("K shape", k.shape)
print("V shape", v.shape)

True
Hidden States: tensor([[[ 0.1335,  0.4780,  0.6428,  ..., -0.0609, -0.2906,  0.9417],
         [ 0.8889,  0.3946,  0.4584,  ...,  0.8159, -0.0048,  0.1393],
         [ 0.9627,  1.1545,  1.2469,  ...,  0.5016,  0.3069,  0.7403],
         ...,
         [ 1.1087,  0.8970,  1.0181,  ...,  0.0492,  0.1162,  0.0941],
         [ 0.4121,  0.6764,  0.8672,  ...,  1.0000,  1.0000,  1.0000],
         [-0.5440, -0.2200,  0.1188,  ...,  1.0000,  1.0000,  1.0000]],

        [[ 0.6316,  0.7004,  0.2591,  ...,  0.4277, -0.7776, -1.7565],
         [ 0.8889,  0.3946,  0.4584,  ...,  0.8159, -0.0048,  0.1393],
         [ 1.1383,  1.6200,  0.6714,  ..., -0.3739, -0.4786,  0.1461],
         ...,
         [ 0.8497,  1.0976,  0.6033,  ...,  0.0663,  0.3130, -0.6179],
         [ 0.4850,  0.4899,  1.1442,  ...,  1.0401,  0.3885,  0.3191],
         [-0.4247, -0.3137,  0.2195,  ...,  0.0492,  0.1162,  0.0941]]])
Hidden States shape: torch.Size([2, 11, 512])
Q shape torch.Size([2, 11, 512])
K shape torch.Siz

* scaling
 

#### FFN

#### Residual / layer norm

---

## 2. Decoder